# 03 - Feature engineering

**Runtime -> Run all. Idempotent** - pure transformation of collected data into
`03_features/fused_v1.parquet`; re-running just rebuilds the same file.

Three rules enforced here:

1. **The n-gram model fits on TRAINING benign domains only** (from the frozen
   `family_disjoint_v1` split). Fitting on the full corpus would leak the test
   distribution into the features - a leak most DGA papers commit silently.
2. **TLS observations are deduplicated, keeping the latest per domain.** A
   ledger-restore bug caused ~6k domains to be probed twice across sessions.
3. **Domains with no certificate stay as rows.** Absence of TLS is signal
   (0.6% of DGA domains have any); inner-joining them away would bias the
   corpus toward reachable hosts.

CT-history features are absent by decision (crt.sh blocks cloud IPs; coverage
0.3% - see `_LEAKAGE_NOTES.md`).

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard xgboost

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from src.utils.io import read_shards
from src.evaluate import splits

INTERIM  = Path(P['data']['interim'])
FEATURES = Path(P['data']['features'])

labels = pd.read_parquet(INTERIM/'domains_labelled.parquet')
split  = splits.load_split(P['data']['splits'], 'family_disjoint_v1')
print('corpus:', labels.shape, '| split hashes verified')

## TLS observations - dedupe

In [ ]:
tls_raw = read_shards(f"{P['data']['collected']}/tls_probe", 'tls_probe')
print('raw rows :', len(tls_raw))

tls = (tls_raw.sort_values('observed_at')
              .drop_duplicates('domain', keep='last')
              .reset_index(drop=True))
print('unique   :', len(tls), f'({len(tls_raw)-len(tls)} duplicates removed, latest kept)')

## n-gram model - training benign only

In [ ]:
from src.features import lexical

train_benign = labels[(labels['label']==0) &
                      labels['domain'].isin(split['domains']['train'])]['domain']
print('fitting n-gram model on', f'{len(train_benign):,}', 'TRAINING benign domains')
lexical.fit_ngram_model(train_benign)
print('fitted:', lexical._NGRAM_MODEL['fitted'],
      '| bigrams:', len(lexical._NGRAM_MODEL['bigram']),
      '| trigrams:', len(lexical._NGRAM_MODEL['trigram']))

## Build the matrix

Lexical features for all 1.78M domains (several minutes of pure Python),
certificate features for the ~30k probed domains, aligned so that unprobed and
certificate-less domains remain rows with `has_certificate=False`.

In [ ]:
from src.features import build as fbuild
import yaml

cfg = yaml.safe_load(open(f'{REPO}/configs/features.yaml'))
quarantined = cfg.get('quarantined', [])

X = fbuild.assemble(labels, tls_df=tls, ct_df=None, dns_df=None,
                    groups=('lexical','certificate'),
                    quarantined=quarantined)
print(X.shape)
print('columns:', list(X.columns))

In [ ]:
# has_certificate must reflect the probe result, not the merge mechanics
print(X.groupby('label')['has_certificate'].mean().round(4))
print()
print('missing-rate of certificate fields (expected high - most domains were')
print('never probed or had no TLS):')
print(X[['validity_days','issuer_org','san_count']].isna().mean().round(3))

In [ ]:
out = FEATURES/'fused_v1.parquet'
X.to_parquet(out, index=False, compression='zstd')
print('wrote', out, X.shape, f'{out.stat().st_size/1e6:.1f} MB')

## Feature-level leakage screen

The deferred half of the audit, now that features exist. Two checks on a 300k
sample: single-feature AUC (a lone feature separating the classes almost
perfectly is the label in disguise) and class-dependent missingness (NaN
patterns that encode the label).

**Expected and acceptable here:** certificate fields WILL show large
missingness gaps - 83% of benign but 0.6% of DGA domains have certificates.
That is the real-world signal this study is about, not an artefact; the
`has_certificate` flag carries it explicitly. The screen exists to catch
anything *else*.

In [ ]:
from src.evaluate import leakage

sample = X.sample(min(300_000, len(X)), random_state=42)
rep = leakage.report(sample)

print('single-feature AUC (top 15):')
display(rep['single_feature_auc'].head(15))
print('CRITICAL (>=0.97):', rep['critical_features'])
print()
print('class-dependent missingness (top 10):')
display(rep['missingness'].head(10))
print()
print(rep['duplicates'])

In [ ]:
findings = rep['single_feature_auc'].head(10).to_string(index=False)
open(P['leakage_notes'], 'a').write(f'''
## Feature-matrix screen ({pd.Timestamp.now().date()}) - fused_v1
Sample: 300k rows. Critical features (AUC >= 0.97): {rep['critical_features'] or 'none'}.
Certificate-field missingness differs by class as expected (83.3% benign vs
0.6% DGA hold certificates); carried explicitly by has_certificate rather than
treated as leakage.
Top single-feature AUCs:
{findings}
''')
print('recorded in _LEAKAGE_NOTES.md')

---

If the critical list is empty (or contains only certificate-presence fields,
which are explained above), the matrix is clean.

**Next:** `05_baselines` - and from there the models. Reminder for evaluation:
random test is 46.9% malicious, family-disjoint test 65.5%, so cross-split
comparisons use prevalence-free metrics (ROC-AUC, FPR@95%TPR); PR-AUC is
reported within-split only.